# **TUẦN 42 — DEEPFASHION-MULTIMODAL ATTRIBUTE RETRIEVAL**

Notebook này tập trung đánh giá khả năng truy xuất trang phục theo thuộc tính chi tiết trên tập dữ liệu DeepFashion-MultiModal, sử dụng checkpoint Attention-Gated Fusion đã huấn luyện ở tuần 41. Quy trình bắt đầu từ khâu kiểm tra môi trường GPU, rà soát tính toàn vẹn của 44.096 ảnh cùng các tệp nhãn (shape, fabric, color-pattern), sau đó nạp mô hình UniFashion kết hợp Attention-Gated Fusion từ checkpoint `best_monitor.pt`. Hệ thống sẽ trích xuất đặc trưng tập gallery, tạo vector truy vấn từ ảnh tham chiếu kèm câu mô tả thuộc tính mục tiêu, rồi tiến hành xếp hạng để tính các chỉ số Recall@1, 5, 10 và 50. Toàn bộ kết quả sẽ được xuất ra file CSV/JSON kèm ảnh minh họa phục vụ cho báo cáo tuần 42.

Vì DeepFashion-MultiModal không cung cấp sẵn bộ benchmark truy xuất chuẩn như FashionIQ, giao thức này do nhóm tự thiết lập và chỉ đóng vai trò là thử nghiệm mở rộng cho khóa luận. Để chạy mượt mà trên GPU T4 của Colab và phòng ngừa sự cố ngắt kết nối, notebook được tích hợp cơ chế lưu cache và checkpoint/resume theo từng phần. Nhờ vậy, các dữ liệu đặc trưng đã xử lý sẽ được tự động tái sử dụng ở các lần chạy sau mà không cần trích xuất lại từ đầu.


## **1 — Phát hiện phần cứng**


In [ ]:
import os
from pathlib import Path

def detect_gpu():
    try:
        import torch
    except Exception as e:
        return {"cuda": False, "error": str(e)}
    if not torch.cuda.is_available():
        return {"cuda": False, "torch": torch.__version__}
    p = torch.cuda.get_device_properties(0)
    return {
        "cuda": True,
        "name": p.name,
        "vram_gib": round(p.total_memory / 2**30, 1),
        "capability": f"{p.major}.{p.minor}",
        "torch": torch.__version__,
    }

HW = detect_gpu()
print(HW)
if not HW.get("cuda"):
    raise RuntimeError("Notebook đánh giá model cần CUDA. Vào Runtime > Change runtime type > GPU T4.")
print(f"GPU OK: {HW['name']} | {HW['vram_gib']} GiB")

{'cuda': True, 'name': 'Tesla T4', 'vram_gib': 14.6, 'capability': '7.5', 'torch': '2.11.0+cu130'}
GPU OK: Tesla T4 | 14.6 GiB


## **2 — Clone đúng repo project và UniFashion upstream**


In [ ]:
import subprocess
from pathlib import Path

def run(cmd, cwd=None):
    print("$", " ".join(map(str, cmd)))
    subprocess.run([str(x) for x in cmd], cwd=str(cwd) if cwd else None, check=True)

PROJECT = Path("/content/khoaluantotnghiep")
UPSTREAM = Path("/content/UniFashion-upstream")

if not PROJECT.exists():
    run(["git", "clone", "--branch", "hien",
         "https://github.com/okok0814/khoaluantotnghiep.git", PROJECT])

if not UPSTREAM.exists():
    run(["git", "clone", "https://github.com/xiangyu-mm/UniFashion.git", UPSTREAM])

assert (PROJECT / "src/unifashion_runtime.py").is_file()
assert (PROJECT / "src/attention_fusion.py").is_file()
assert (PROJECT / "src/unifashion_training.py").is_file()

alias = Path("/content/UniFashion")
if not alias.exists():
    alias.symlink_to(UPSTREAM, target_is_directory=True)

head = subprocess.run(["git", "-C", str(PROJECT), "rev-parse", "HEAD"],
                      capture_output=True, text=True, check=True).stdout.strip()
print("PROJECT:", PROJECT)
print("PROJECT HEAD:", head)
print("UPSTREAM:", UPSTREAM)

PROJECT: /content/khoaluantotnghiep
PROJECT HEAD: 1d64db4b87ae87c2cf95d970d902908680360e4c
UPSTREAM: /content/UniFashion-upstream


## **3 — Mount Google Drive và khai báo đường dẫn**


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive")
KHOALUAN = DRIVE / "KHOALUAN"

DEEP_ROOT = KHOALUAN / "deepfashion_mm"
IMAGES_DIR = DEEP_ROOT / "images"
CAPTIONS_JSON = DEEP_ROOT / "captions.json"

SHAPE_TXT = DEEP_ROOT / "labels" / "labels" / "shape" / "shape_anno_all.txt"
FABRIC_TXT = DEEP_ROOT / "labels" / "labels" / "texture" / "fabric_ann.txt"
PATTERN_TXT = DEEP_ROOT / "labels" / "labels" / "texture" / "pattern_ann.txt"

RUN41 = KHOALUAN / "Tuần 41" / "UniFashionResults_run2"
BEST_CKPT = RUN41 / "best_monitor.pt"
LAST_CKPT = RUN41 / "last.pt"
ENV41 = RUN41 / "environment.json"
CONFIG41 = RUN41 / "config.json"

ASSETS = DRIVE / "UniFashionAssets"

OUT42 = KHOALUAN / "Tuần 42" / "DeepFashionAttributeRetrieval"
OUT42.mkdir(parents=True, exist_ok=True)

paths = {
    "DEEP_ROOT": DEEP_ROOT,
    "IMAGES_DIR": IMAGES_DIR,
    "CAPTIONS_JSON": CAPTIONS_JSON,
    "SHAPE_TXT": SHAPE_TXT,
    "FABRIC_TXT": FABRIC_TXT,
    "PATTERN_TXT": PATTERN_TXT,
    "BEST_CKPT": BEST_CKPT,
    "LAST_CKPT": LAST_CKPT,
    "ENV41": ENV41,
    "CONFIG41": CONFIG41,
    "ASSETS": ASSETS,
    "OUT42": OUT42,
}
for k, p in paths.items():
    print(f"{k:14s} -> {p}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DEEP_ROOT      -> /content/drive/MyDrive/KHOALUAN/deepfashion_mm
IMAGES_DIR     -> /content/drive/MyDrive/KHOALUAN/deepfashion_mm/images
CAPTIONS_JSON  -> /content/drive/MyDrive/KHOALUAN/deepfashion_mm/captions.json
SHAPE_TXT      -> /content/drive/MyDrive/KHOALUAN/deepfashion_mm/labels/labels/shape/shape_anno_all.txt
FABRIC_TXT     -> /content/drive/MyDrive/KHOALUAN/deepfashion_mm/labels/labels/texture/fabric_ann.txt
PATTERN_TXT    -> /content/drive/MyDrive/KHOALUAN/deepfashion_mm/labels/labels/texture/pattern_ann.txt
BEST_CKPT      -> /content/drive/MyDrive/KHOALUAN/Tuần 41/UniFashionResults_run2/best_monitor.pt
LAST_CKPT      -> /content/drive/MyDrive/KHOALUAN/Tuần 41/UniFashionResults_run2/last.pt
ENV41          -> /content/drive/MyDrive/KHOALUAN/Tuần 41/UniFashionResults_run2/environment.json
CONFIG41       -> /content/drive/MyDrive/KHOALUAN/Tuần 41/UniF

### **3.1 — Kiểm tra file bắt buộc trước khi tốn GPU**


In [ ]:
required = [
    IMAGES_DIR, CAPTIONS_JSON, SHAPE_TXT, FABRIC_TXT, PATTERN_TXT,
    BEST_CKPT, ENV41, CONFIG41,
    ASSETS / "eva_vit_g.pth",
    ASSETS / "pretrain" / "none_lora_0.pt",
    ASSETS / "bert-base-uncased",
]
missing = [str(p) for p in required if not p.exists()]
for p in required:
    print(("OK      " if p.exists() else "MISSING "), p)
if missing:
    raise FileNotFoundError("Thiếu dữ liệu/model asset:\n" + "\n".join(missing))
print("\nPASS: đủ đường dẫn bắt buộc")

OK       /content/drive/MyDrive/KHOALUAN/deepfashion_mm/images
OK       /content/drive/MyDrive/KHOALUAN/deepfashion_mm/captions.json
OK       /content/drive/MyDrive/KHOALUAN/deepfashion_mm/labels/labels/shape/shape_anno_all.txt
OK       /content/drive/MyDrive/KHOALUAN/deepfashion_mm/labels/labels/texture/fabric_ann.txt
OK       /content/drive/MyDrive/KHOALUAN/deepfashion_mm/labels/labels/texture/pattern_ann.txt
OK       /content/drive/MyDrive/KHOALUAN/Tuần 41/UniFashionResults_run2/best_monitor.pt
OK       /content/drive/MyDrive/KHOALUAN/Tuần 41/UniFashionResults_run2/environment.json
OK       /content/drive/MyDrive/KHOALUAN/Tuần 41/UniFashionResults_run2/config.json
OK       /content/drive/MyDrive/UniFashionAssets/eva_vit_g.pth
OK       /content/drive/MyDrive/UniFashionAssets/pretrain/none_lora_0.pt
OK       /content/drive/MyDrive/UniFashionAssets/bert-base-uncased

PASS: đủ đường dẫn bắt buộc


## **4 — Tạo môi trường UniFashion giống workflow Tuần 41**

Project dùng môi trường ghim riêng để tránh xung đột giữa Colab mới và dependency cũ của UniFashion.

Cell này cài `uv`, tạo Python 3.12 riêng, cài PyTorch CUDA và requirements của project, không thay Python mặc định của Colab.


In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

REQ = PROJECT / "requirements-unifashion.txt"
if not REQ.is_file():
    raise FileNotFoundError(f"Thiếu {REQ}")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
UV = shutil.which("uv")
if not UV:
    raise RuntimeError("Không tìm thấy uv sau khi cài.")

subprocess.run([UV, "python", "install", "3.12"], check=True)

VENV = Path("/content/unifashion-py312")
PYTHON = VENV / "bin" / "python"
if not PYTHON.is_file():
    subprocess.run([UV, "venv", "--python", "3.12", str(VENV)], check=True)

# Giữ cùng cặp torch/torchvision notebook Tuần 41 sử dụng.
subprocess.run([
    UV, "pip", "install", "--python", str(PYTHON),
    "--index-url", "https://download.pytorch.org/whl/cu124",
    "--extra-index-url", "https://pypi.org/simple",
    "torch==2.6.0", "torchvision==0.21.0"
], check=True)

subprocess.run([
    UV, "pip", "install", "--python", str(PYTHON),
    "-r", str(REQ)
], check=True)

check = subprocess.run(
    [str(PYTHON), "-c",
     "import torch, transformers, timm, pandas, PIL; "
     "print('python env OK'); print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"],
    capture_output=True, text=True, check=True
)
print(check.stdout)

python env OK
torch 2.6.0+cu124 cuda True



## **5 — Kiểm tra source hiện tại có khớp source đã train hay không**


In [ ]:
import hashlib, json
from pathlib import Path

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

env41 = json.loads(ENV41.read_text(encoding="utf-8"))
expected_hashes = env41.get("fingerprint", {}).get("code", {})

audit_code = {}
for name, expected in expected_hashes.items():
    candidates = [PROJECT / "src" / name, PROJECT / "scripts" / name]
    p = next((x for x in candidates if x.is_file()), None)
    if p is None:
        audit_code[name] = {"status": "missing"}
        continue
    actual = sha256(p)
    audit_code[name] = {
        "path": str(p),
        "expected": expected,
        "actual": actual,
        "match": actual == expected,
    }

for name, item in audit_code.items():
    print(name, "MATCH" if item.get("match") else item.get("status", "DIFFERENT"))

mismatch = [k for k,v in audit_code.items() if v.get("match") is False]
if mismatch:
    raise RuntimeError(
        "Source hiện tại khác source ghi trong environment.json của lượt train: "
        + ", ".join(mismatch)
    )
print("\nPASS source fingerprint khớp lượt train Tuần 41")

train_unifashion_finetune.py MATCH
unifashion_training.py MATCH
unifashion_runtime.py MATCH
unifashion_sanity_data.py MATCH
fashioniq_unifashion_dataset.py MATCH
attention_fusion.py MATCH
train_unifashion_sanity.py MATCH

PASS source fingerprint khớp lượt train Tuần 41


## **6 — Kiểm tra DeepFashion-MultiModal**

Theo bộ dữ liệu chính thức:
- ảnh: 44.096,
- `fabric_ann.txt`: 3 nhãn cho upper/lower/outer,
- `pattern_ann.txt`: 3 nhãn color/pattern cho upper/lower/outer,
- `shape_anno_all.txt`: 12 nhãn shape,
- `captions.json`: mô tả văn bản theo tên ảnh.



In [ ]:
import json, os
from pathlib import Path

image_files = sorted(
    p for p in IMAGES_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)
image_names = {p.name for p in image_files}
captions = json.loads(CAPTIONS_JSON.read_text(encoding="utf-8"))

def nonempty_lines(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

shape_lines = nonempty_lines(SHAPE_TXT)
fabric_lines = nonempty_lines(FABRIC_TXT)
pattern_lines = nonempty_lines(PATTERN_TXT)

summary = {
    "images": len(image_files),
    "captions": len(captions),
    "shape_rows": len(shape_lines),
    "fabric_rows": len(fabric_lines),
    "pattern_rows": len(pattern_lines),
}
print(json.dumps(summary, indent=2, ensure_ascii=False))

if len(image_files) != 44096:
    print(f"CẢNH BÁO: images thực tế = {len(image_files)}, khác 44.096.")
if len(fabric_lines) != 44096:
    print(f"CẢNH BÁO: fabric rows = {len(fabric_lines)}, khác 44.096.")
if len(pattern_lines) != 44096:
    print(f"CẢNH BÁO: pattern rows = {len(pattern_lines)}, khác 44.096.")

caption_names = set(captions)
print("captions thiếu ảnh thực tế:", len(caption_names - image_names))
print("ảnh thiếu caption:", len(image_names - caption_names))

{
  "images": 44114,
  "captions": 42544,
  "shape_rows": 42544,
  "fabric_rows": 44096,
  "pattern_rows": 44096
}
CẢNH BÁO: images thực tế = 44114, khác 44.096.
captions thiếu ảnh thực tế: 0
ảnh thiếu caption: 1570


### **6.1 — Giải mã nhãn theo định nghĩa chính thức**

In [ ]:
# Mapping từ README chính thức DeepFashion-MultiModal.
SLEEVE = {
    0: "sleeveless", 1: "short-sleeve", 2: "medium-sleeve",
    3: "long-sleeve", 4: "not-long-sleeve", 5: "NA",
}
LOWER_LENGTH = {0: "three-point", 1: "medium-short", 2: "three-quarter", 3: "long", 4: "NA"}
SOCKS = {0: "no", 1: "socks", 2: "leggings", 3: "NA"}
HAT = {0: "no", 1: "yes", 2: "NA"}
GLASSES = {0: "no", 1: "eyeglasses", 2: "sunglasses", 3: "glasses-in-hand-or-clothes", 4: "NA"}
NECKWEAR = {0: "no", 1: "yes", 2: "NA"}
WRIST = {0: "no", 1: "yes", 2: "NA"}
RING = {0: "no", 1: "yes", 2: "NA"}
WAIST = {0: "no", 1: "belt", 2: "clothing-at-waist", 3: "hidden", 4: "NA"}
NECKLINE = {0: "v-shape", 1: "square", 2: "round", 3: "standing", 4: "lapel", 5: "suspenders", 6: "NA"}
CARDIGAN = {0: "yes", 1: "no", 2: "NA"}
COVER_NAVEL = {0: "no", 1: "yes", 2: "NA"}

FABRIC = {
    0: "denim", 1: "cotton", 2: "leather", 3: "furry",
    4: "knitted", 5: "chiffon", 6: "other", 7: "NA",
}
PATTERN = {
    0: "floral", 1: "graphic", 2: "striped", 3: "pure-color",
    4: "lattice", 5: "other", 6: "color-block", 7: "NA",
}

print("Mappings loaded.")

Mappings loaded.


### **6.2 — Đọc annotation thành bảng**


In [ ]:
import pandas as pd
import numpy as np

shape_cols = [
    "image", "sleeve", "lower_length", "socks", "hat", "glasses", "neckwear",
    "wrist", "ring", "waist", "neckline", "cardigan", "cover_navel"
]
fabric_cols = ["image", "upper_fabric", "lower_fabric", "outer_fabric"]
pattern_cols = ["image", "upper_pattern", "lower_pattern", "outer_pattern"]

def parse_table(lines, columns):
    rows = []
    for line in lines:
        parts = line.split()
        if len(parts) != len(columns):
            continue
        rows.append([str(parts[0])] + [int(x) for x in parts[1:]])
    df = pd.DataFrame(rows, columns=columns)
    df["image"] = df["image"].astype(str)
    return df

shape_df = parse_table(shape_lines, shape_cols)
fabric_df = parse_table(fabric_lines, fabric_cols)
pattern_df = parse_table(pattern_lines, pattern_cols)

caption_names = set(map(str, captions.keys()))
shape_names = set(shape_df["image"].astype(str))
fabric_names = set(fabric_df["image"].astype(str))
pattern_names = set(pattern_df["image"].astype(str))
image_names = set(map(str, image_names))

usable_images = sorted(
    image_names & caption_names & shape_names & fabric_names & pattern_names
)

print("Tổng ảnh vật lý ", len(image_names))
print("Ảnh có caption ", len(caption_names))
print("Ảnh có shape ", len(shape_names))
print("Ảnh có fabric ", len(fabric_names))
print("Ảnh có pattern ", len(pattern_names))
print("Ảnh dùng được cuối cùng ", len(usable_images))

base = pd.DataFrame({"image": usable_images})
base["image"] = base["image"].astype(str)

df = (
    base
    .merge(fabric_df, on="image", how="left")
    .merge(pattern_df, on="image", how="left")
    .merge(shape_df, on="image", how="left")
)

df["caption"] = df["image"].map(captions)

# lấy gender và category từ tên file
df["gender"] = df["image"].astype(str).str.extract(r"^(MEN|WOMEN)-", expand=False)
df["category"] = df["image"].astype(str).str.extract(r"^(?:MEN|WOMEN)-(.+?)-id_", expand=False)

print("\nKích thước dataframe cuối:", df.shape)
display(df.head(5))

Tổng ảnh vật lý  44114
Ảnh có caption  42544
Ảnh có shape  42544
Ảnh có fabric  44096
Ảnh có pattern  44096
Ảnh dùng được cuối cùng  42544

Kích thước dataframe cuối: (42544, 22)


,image,upper_fabric,lower_fabric,outer_fabric,upper_pattern,lower_pattern,outer_pattern,sleeve,lower_length,socks,...,neckwear,wrist,ring,waist,neckline,cardigan,cover_navel,caption,gender,category
0,MEN-Denim-id_00000080-01_7_additional.jpg,1,1,7,3,4,7,5,3,0,...,0,0,0,3,2,1,1,The lower clothing is of long length. The fabr...,MEN,Denim
1,MEN-Denim-id_00000089-01_7_additional.jpg,1,1,7,3,3,7,0,3,0,...,0,0,0,3,2,1,1,"His tank top has sleeves cut off, cotton fabri...",MEN,Denim
2,MEN-Denim-id_00000089-02_7_additional.jpg,1,1,7,2,3,7,3,3,0,...,0,0,0,3,4,1,1,"His sweater has long sleeves, cotton fabric an...",MEN,Denim
3,MEN-Denim-id_00000089-03_7_additional.jpg,1,1,7,3,3,7,1,3,0,...,0,0,0,3,2,1,1,"His shirt has short sleeves, cotton fabric and...",MEN,Denim
4,MEN-Denim-id_00000089-04_7_additional.jpg,0,1,7,3,3,7,3,3,0,...,0,0,0,3,4,1,1,"The sweater the person wears has long sleeves,...",MEN,Denim


### **6.3 — Kiểm tra thiếu nhãn và lưu data audit**


In [ ]:
audit = {
    "counts": summary,
    "missing": {
        "caption": int(df["caption"].isna().sum()),
        "shape": int(df["sleeve"].isna().sum()),
        "fabric": int(df["upper_fabric"].isna().sum()),
        "pattern": int(df["upper_pattern"].isna().sum()),
    },
    "unique_categories": sorted(x for x in df["category"].dropna().unique().tolist()),
    "note": "shape annotations may cover fewer images than fabric/pattern; no missing shape label is invented.",
}
(OUT42 / "data_audit.json").write_text(
    json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8"
)
df.to_csv(OUT42 / "deepfashion_annotations_merged.csv", index=False)

print(json.dumps(audit, ensure_ascii=False, indent=2))
print("Saved:", OUT42 / "data_audit.json")

{
  "counts": {
    "images": 44114,
    "captions": 42544,
    "shape_rows": 42544,
    "fabric_rows": 44096,
    "pattern_rows": 44096
  },
  "missing": {
    "caption": 0,
    "shape": 0,
    "fabric": 0,
    "pattern": 0
  },
  "unique_categories": [
    "Blouses_Shirts",
    "Cardigans",
    "Denim",
    "Dresses",
    "Graphic_Tees",
    "Jackets_Coats",
    "Jackets_Vests",
    "Leggings",
    "Pants",
    "Rompers_Jumpsuits",
    "Shirts_Polos",
    "Shorts",
    "Skirts",
    "Suiting",
    "Sweaters",
    "Sweatshirts_Hoodies",
    "Tees_Tanks"
  ],
  "note": "shape annotations may cover fewer images than fabric/pattern; no missing shape label is invented."
}
Saved: /content/drive/MyDrive/KHOALUAN/Tuần 42/DeepFashionAttributeRetrieval/data_audit.json


## **7 — Xây giao thức retrieval thuộc tính**

Do tập DeepFashion-MultiModal không có sẵn cấu trúc bộ ba dữ liệu (triplet) như FashionIQ, nhóm đã thiết kế một giao thức thử nghiệm riêng mang tên **multi-positive attribute retrieval**.

Mô hình UniFashion Attention-Gated Fusion sẽ nhận vào một ảnh tham chiếu (reference image) và câu mô tả thay đổi (query text) được trích xuất trực tiếp từ nhãn thuộc tính của ảnh đích (target seed)

Ví dụ: "long-sleeve, round neckline, cotton upper fabric, striped upper pattern".

Trong giao thức này, đáp án đúng (ground truth) được định nghĩa là toàn bộ các ảnh trong gallery có cùng danh mục và đáp ứng chính xác bộ thuộc tính mục tiêu. Sau khi loại bỏ ảnh tham chiếu khỏi danh sách xếp hạng, một lượt truy vấn được tính là thành công (hit@K) nếu tìm thấy ít nhất một ảnh đáp án đúng nằm trong Top-K kết quả đầu tiên.


### **7.1 — Chọn chế độ screening / full**


In [ ]:
# full: dùng toàn bộ gallery hợp lệ; có checkpoint/resume để an toàn khi Colab ngắt.
EVAL_MODE = "full"

SEED = 42
if EVAL_MODE == "screening":
    MAX_GALLERY = 3000
    MAX_QUERIES_PER_TIER = 120
    BATCH_SIZE = 2
else:
    MAX_GALLERY = None       # toàn bộ ảnh hợp lệ
    MAX_QUERIES_PER_TIER = 1000
    BATCH_SIZE = 2

# Resume granularity:
# Gallery được lưu mỗi 256 ảnh.
# Query được lưu mỗi 20 query.
# Nếu Colab ngắt, lần chạy sau chỉ làm tiếp chunk chưa hoàn tất.
GALLERY_CHUNK_SIZE = 256
QUERY_CHUNK_SIZE = 20

KS = (1, 5, 10, 50)
print("EVAL_MODE =", EVAL_MODE)
print("MAX_GALLERY =", MAX_GALLERY)
print("MAX_QUERIES_PER_TIER =", MAX_QUERIES_PER_TIER)
print("BATCH_SIZE =", BATCH_SIZE)
print("GALLERY_CHUNK_SIZE =", GALLERY_CHUNK_SIZE)
print("QUERY_CHUNK_SIZE =", QUERY_CHUNK_SIZE)

EVAL_MODE = full
MAX_GALLERY = None
MAX_QUERIES_PER_TIER = 1000
BATCH_SIZE = 2
GALLERY_CHUNK_SIZE = 256
QUERY_CHUNK_SIZE = 20


### **7.2 — Tạo các mức độ query thuộc tính**


In [ ]:
import random, hashlib

def attr_text(row, tier):
    parts = []
    if tier in {"fabric_pattern", "sleeve_fabric_pattern", "detailed"}:
        uf = int(row.upper_fabric)
        up = int(row.upper_pattern)
        if FABRIC.get(uf) == "NA" or PATTERN.get(up) == "NA":
            return None
        parts += [f"upper fabric {FABRIC[uf]}", f"upper pattern {PATTERN[up]}"]

    if tier in {"sleeve_fabric_pattern", "detailed"}:
        sl = int(row.sleeve)
        if SLEEVE.get(sl) == "NA":
            return None
        parts.insert(0, f"{SLEEVE[sl]} upper clothing")

    if tier == "detailed":
        nl = int(row.neckline)
        if NECKLINE.get(nl) == "NA":
            return None
        parts.insert(1, f"{NECKLINE[nl]} neckline")

    return "Find clothing with " + ", ".join(parts) + "."

TIERS = {
    "fabric_pattern": ["upper_fabric", "upper_pattern"],
    "sleeve_fabric_pattern": ["sleeve", "upper_fabric", "upper_pattern"],
    "detailed": ["sleeve", "neckline", "upper_fabric", "upper_pattern"],
}

valid = df.dropna(subset=["gender", "category", "upper_fabric", "upper_pattern"]).copy()
valid = valid[valid["upper_fabric"].astype(int) != 7]
valid = valid[valid["upper_pattern"].astype(int) != 7]

rng = np.random.default_rng(SEED)
if MAX_GALLERY is not None and len(valid) > MAX_GALLERY:
    # Dữ liệu được lấy mẫu theo từng nhóm giới tính và loại trang phục nhằm giữ sự phân bố tương đối giữa các nhóm
    parts = []
    groups = list(valid.groupby(["gender", "category"]))
    per = max(20, MAX_GALLERY // max(1, len(groups)))
    for _, g in groups:
        n = min(len(g), per)
        parts.append(g.sample(n=n, random_state=SEED))
    gallery_df = pd.concat(parts).drop_duplicates("image")
    if len(gallery_df) > MAX_GALLERY:
        gallery_df = gallery_df.sample(MAX_GALLERY, random_state=SEED)
else:
    gallery_df = valid.copy()

gallery_df = gallery_df.reset_index(drop=True)
gallery_names = gallery_df["image"].tolist()
gallery_index = {name:i for i,name in enumerate(gallery_names)}
print("gallery size ", len(gallery_df))

gallery size  41571


### **7.3 — Sinh query + ground truth một cách xác định**


In [ ]:
import os
import numpy as np
import pandas as pd


def make_queries(gallery_df, tier, max_queries, seed=42):
    cols = TIERS[tier]

    required = list(
        dict.fromkeys(
            ["image", "gender", "category", "sleeve", "neckline"]
            + cols
        )
    )

    work = gallery_df.dropna(subset=required).copy()

    # Chuẩn hóa dtype trước khi so sánh.
    for c in ["sleeve", "neckline", "upper_fabric", "upper_pattern"]:
        if c in work.columns:
            work[c] = pd.to_numeric(work[c], errors="coerce")

    work = work.dropna(
        subset=[
            c
            for c in [
                "sleeve",
                "neckline",
                "upper_fabric",
                "upper_pattern",
            ]
            if c in work.columns
        ]
    )

    # Bỏ các mã NA theo mapping hiện tại của notebook.
    if "sleeve" in cols:
        work = work[work["sleeve"].astype(int) != 5]

    if "neckline" in cols:
        work = work[work["neckline"].astype(int) != 6]

    work = work[
        work["upper_fabric"].astype(int) != 7
    ]

    work = work[
        work["upper_pattern"].astype(int) != 7
    ]

    print(
        f"[{tier}] candidates sau filter:",
        len(work)
    )

    if work.empty:
        return pd.DataFrame(
            columns=[
                "tier",
                "reference",
                "target_seed",
                "text",
                "gender",
                "category",
                "sleeve",
                "neckline",
                "upper_fabric",
                "upper_pattern",
                "positive_count",
            ]
        )

    rng = np.random.default_rng(seed)

    # Shuffle deterministic.
    work = work.sample(
        frac=1,
        random_state=seed
    ).reset_index(drop=True)

    rows = []

    for _, target in work.iterrows():

        target_gender = str(target["gender"])
        target_category = str(target["category"])

        # Ground truth positive set
        # Phải KHỚP logic evaluator:
        # gender + category + exact queried attrs
        positive_mask = (
            gallery_df["gender"].astype(str).eq(target_gender)
            &
            gallery_df["category"].astype(str).eq(target_category)
        )

        for c in cols:
            target_value = int(target[c])

            numeric_col = pd.to_numeric(
                gallery_df[c],
                errors="coerce"
            )

            positive_mask &= numeric_col.eq(
                target_value
            )

        positives = gallery_df[
            positive_mask
        ].copy()

        if positives.empty:
            continue

        # Reference pool
        # Cùng gender/category để composition có ý nghĩa,
        # nhưng KHÁC target và khác ít nhất một queried attribute.
        same_category = gallery_df[
            gallery_df["gender"].astype(str).eq(target_gender)
            &
            gallery_df["category"].astype(str).eq(target_category)
            &
            gallery_df["image"].astype(str).ne(
                str(target["image"])
            )
        ].copy()

        if same_category.empty:
            continue

        diff_mask = np.zeros(
            len(same_category),
            dtype=bool
        )

        for c in cols:
            ref_values = pd.to_numeric(
                same_category[c],
                errors="coerce"
            )

            diff_mask |= (
                ref_values.to_numpy()
                != int(target[c])
            )

        refs = same_category[
            diff_mask
        ].copy()

        if refs.empty:
            continue

        ref = refs.iloc[
            int(
                rng.integers(
                    0,
                    len(refs)
                )
            )
        ]

        # Reference đang được evaluator loại khỏi ranking.
        # Nếu reference tình cờ nằm trong positive set thì trừ nó ra.
        positive_names = set(
            positives["image"].astype(str)
        )

        positive_names.discard(
            str(ref["image"])
        )

        positive_count = len(
            positive_names
        )

        # Cần ít nhất 1 đáp án thật sau khi loại reference.
        if positive_count < 1:
            continue

        text = attr_text(
            target,
            tier
        )

        if not text:
            continue

        rows.append(
            {
                "tier": tier,
                "reference": str(
                    ref["image"]
                ),
                "target_seed": str(
                    target["image"]
                ),
                "text": text,
                "gender": target_gender,
                "category": target_category,
                "sleeve": int(
                    target["sleeve"]
                ),
                "neckline": int(
                    target["neckline"]
                ),
                "upper_fabric": int(
                    target["upper_fabric"]
                ),
                "upper_pattern": int(
                    target["upper_pattern"]
                ),
                "positive_count": positive_count,
            }
        )

        if len(rows) >= max_queries:
            break

    columns = [
        "tier",
        "reference",
        "target_seed",
        "text",
        "gender",
        "category",
        "sleeve",
        "neckline",
        "upper_fabric",
        "upper_pattern",
        "positive_count",
    ]

    result = pd.DataFrame(
        rows,
        columns=columns
    )

    print(
        f"[{tier}] queries tạo được:",
        len(result)
    )

    if len(result):
        print(
            f"[{tier}] positive_count min/median/max:",
            int(result["positive_count"].min()),
            float(result["positive_count"].median()),
            int(result["positive_count"].max()),
        )

    return result

# TẠO QUERY CHO 3 TIER
query_parts = []
expected_by_tier = {}

for i, tier in enumerate(TIERS):

    q = make_queries(
        gallery_df,
        tier,
        MAX_QUERIES_PER_TIER,
        seed=SEED + i,
    )

    expected_by_tier[tier] = len(q)

    print(
        tier,
        "queries:",
        len(q)
    )

    if q.empty:
        raise RuntimeError(
            f"Không tạo được query cho tier={tier}. "
            f"gallery_size={len(gallery_df)}. "
            "Kiểm tra annotation của tier này trước evaluation."
        )

    query_parts.append(q)


queries = pd.concat(
    query_parts,
    ignore_index=True
)

if queries.empty:
    raise RuntimeError(
        "Query table rỗng. "
        "Không được phép chạy evaluation."
    )

# SANITY CHECK
gallery_name_set = set(
    gallery_df["image"].astype(str)
)

if not set(
    queries["target_seed"]
).issubset(
    gallery_name_set
):
    raise RuntimeError(
        "Có target_seed nằm ngoài gallery."
    )

if (
    queries["reference"]
    == queries["target_seed"]
).any():
    raise RuntimeError(
        "Có query reference == target_seed."
    )

if (
    queries["text"]
    .astype(str)
    .str.strip()
    .eq("")
    .any()
):
    raise RuntimeError(
        "Có query text rỗng."
    )


print("\n===== QUERY SUMMARY =====")

print(
    queries.groupby("tier")
    .size()
)

print(
    "\nTotal queries:",
    len(queries)
)

# GHI CSV ATOMIC
query_path = (
    OUT42
    / f"queries_{EVAL_MODE}.csv"
)

query_tmp = query_path.with_suffix(
    query_path.suffix + ".tmp"
)

queries.to_csv(
    query_tmp,
    index=False
)


# Đọc lại file temp để xác minh trước replace.
check_queries = pd.read_csv(
    query_tmp
)

required_query_cols = {
    "tier",
    "reference",
    "target_seed",
    "text",
    "gender",
    "category",
    "sleeve",
    "neckline",
    "upper_fabric",
    "upper_pattern",
    "positive_count",
}

if len(check_queries) != len(queries):
    raise RuntimeError(
        "Query CSV temp thiếu dòng: "
        f"expected={len(queries)}, "
        f"read_back={len(check_queries)}"
    )

missing_cols = (
    required_query_cols
    - set(check_queries.columns)
)

if missing_cols:
    raise RuntimeError(
        "Query CSV temp thiếu cột: "
        f"{sorted(missing_cols)}"
    )

if query_tmp.stat().st_size == 0:
    raise RuntimeError(
        "Query CSV temp bị 0 byte."
    )

os.replace(
    query_tmp,
    query_path
)

# PROTOCOL
protocol = {
    "name":
        "DeepFashion-MultiModal custom multi-positive attribute retrieval",

    "official_benchmark":
        False,

    "evaluation_mode":
        EVAL_MODE,

    "seed":
        SEED,

    "gallery_size":
        len(gallery_df),

    "query_count":
        len(queries),

    "queries_per_tier":
        expected_by_tier,

    "tiers":
        TIERS,

    "ks":
        list(KS),

    "reference_excluded":
        True,

    "positive_definition":
        (
            "same gender/category and exact match "
            "on all queried official annotation fields"
        ),

    "query_storage":
        (
            "compact attribute codes; positive set "
            "is reconstructed from gallery during evaluation"
        ),
}


protocol_path = (
    OUT42
    / f"protocol_{EVAL_MODE}.json"
)

protocol_tmp = protocol_path.with_suffix(
    protocol_path.suffix + ".tmp"
)

protocol_tmp.write_text(
    json.dumps(
        protocol,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
)

os.replace(
    protocol_tmp,
    protocol_path
)


display(
    queries.head(10)
)

print(
    "\ntotal queries:",
    len(queries)
)

print(
    "query CSV bytes:",
    query_path.stat().st_size
)

print(
    "Saved atomically:",
    query_path
)

[fabric_pattern] candidates sau filter: 41571
[fabric_pattern] queries tạo được: 1000
[fabric_pattern] positive_count min/median/max: 1 858.0 6607
fabric_pattern queries: 1000
[sleeve_fabric_pattern] candidates sau filter: 41487
[sleeve_fabric_pattern] queries tạo được: 1000
[sleeve_fabric_pattern] positive_count min/median/max: 1 404.0 3441
sleeve_fabric_pattern queries: 1000
[detailed] candidates sau filter: 26818
[detailed] queries tạo được: 1000
[detailed] positive_count min/median/max: 1 107.0 1082
detailed queries: 1000

===== QUERY SUMMARY =====
tier
detailed                 1000
fabric_pattern           1000
sleeve_fabric_pattern    1000
dtype: int64

Total queries: 3000


,tier,reference,target_seed,text,gender,category,sleeve,neckline,upper_fabric,upper_pattern,positive_count
0,fabric_pattern,WOMEN-Blouses_Shirts-id_00000737-01_3_back.jpg,WOMEN-Blouses_Shirts-id_00004060-02_7_addition...,"Find clothing with upper fabric chiffon, upper...",WOMEN,Blouses_Shirts,3,6,5,3,1702
1,fabric_pattern,MEN-Tees_Tanks-id_00005882-02_4_full.jpg,MEN-Tees_Tanks-id_00005971-01_2_side.jpg,"Find clothing with upper fabric cotton, upper ...",MEN,Tees_Tanks,3,3,1,3,1323
2,fabric_pattern,MEN-Tees_Tanks-id_00005056-01_7_additional.jpg,MEN-Tees_Tanks-id_00003988-01_1_front.jpg,"Find clothing with upper fabric cotton, upper ...",MEN,Tees_Tanks,1,2,1,6,245
3,fabric_pattern,WOMEN-Tees_Tanks-id_00003500-05_4_full.jpg,WOMEN-Tees_Tanks-id_00005735-03_2_side.jpg,"Find clothing with upper fabric cotton, upper ...",WOMEN,Tees_Tanks,3,6,1,3,6607
4,fabric_pattern,WOMEN-Rompers_Jumpsuits-id_00002871-02_3_back.jpg,WOMEN-Rompers_Jumpsuits-id_00005043-04_3_back.jpg,"Find clothing with upper fabric cotton, upper ...",WOMEN,Rompers_Jumpsuits,0,6,1,3,641
5,fabric_pattern,WOMEN-Tees_Tanks-id_00006871-04_3_back.jpg,WOMEN-Tees_Tanks-id_00005976-03_4_full.jpg,"Find clothing with upper fabric cotton, upper ...",WOMEN,Tees_Tanks,0,2,1,1,2007
6,fabric_pattern,WOMEN-Blouses_Shirts-id_00000806-02_2_side.jpg,WOMEN-Blouses_Shirts-id_00002858-02_7_addition...,"Find clothing with upper fabric cotton, upper ...",WOMEN,Blouses_Shirts,3,6,1,3,1298
7,fabric_pattern,WOMEN-Jackets_Coats-id_00006460-01_1_front.jpg,WOMEN-Jackets_Coats-id_00007941-02_3_back.jpg,"Find clothing with upper fabric cotton, upper ...",WOMEN,Jackets_Coats,3,6,1,3,858
8,fabric_pattern,WOMEN-Graphic_Tees-id_00001143-01_3_back.jpg,WOMEN-Graphic_Tees-id_00007242-01_2_side.jpg,"Find clothing with upper fabric cotton, upper ...",WOMEN,Graphic_Tees,0,2,1,1,842
9,fabric_pattern,WOMEN-Dresses-id_00000731-12_4_full.jpg,WOMEN-Dresses-id_00006868-02_4_full.jpg,"Find clothing with upper fabric other, upper p...",WOMEN,Dresses,0,5,6,3,10



total queries: 3000
query CSV bytes: 673262
Saved atomically: /content/drive/MyDrive/KHOALUAN/Tuần 42/DeepFashionAttributeRetrieval/queries_full.csv


## **8 — Kiểm tra checkpoint Tuần 41 trước khi load model**


In [ ]:
# Chạy bằng Python môi trường riêng để tránh torch version mismatch.
checkpoint_audit_script = r'''
import json, sys, torch
from pathlib import Path

ckpt = Path(sys.argv[1])
saved = torch.load(ckpt, map_location="cpu", weights_only=True, mmap=True)
state = saved.get("model", {})
fusion = sorted(k for k in state if k.startswith("attention_fusion."))

report = {
    "checkpoint": str(ckpt),
    "keys": sorted(saved.keys()),
    "epoch": saved.get("epoch"),
    "monitor_loss": saved.get("monitor_loss"),
    "selection": saved.get("selection"),
    "model_keys": len(state),
    "attention_fusion_keys": fusion,
    "attention_fusion_present": bool(fusion),
}
print(json.dumps(report, indent=2))
if not state:
    raise RuntimeError("Checkpoint không có model state.")
if not fusion:
    raise RuntimeError("Checkpoint không có attention_fusion weights.")
'''

audit_file = Path("/content/check_best_monitor.py")
audit_file.write_text(checkpoint_audit_script, encoding="utf-8")
res = subprocess.run(
    [str(PYTHON), str(audit_file), str(BEST_CKPT)],
    capture_output=True, text=True
)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError("Checkpoint audit thất bại.")
(OUT42 / "checkpoint_audit.txt").write_text(res.stdout, encoding="utf-8")

{
  "checkpoint": "/content/drive/MyDrive/KHOALUAN/Tu\u1ea7n 41/UniFashionResults_run2/best_monitor.pt",
  "keys": [
    "config",
    "epoch",
    "model",
    "monitor_loss",
    "selection"
  ],
  "epoch": 8,
  "monitor_loss": 0.31285399156263777,
  "selection": "fixed monitor loss, NOT Recall@K",
  "model_keys": 359,
  "attention_fusion_keys": [
    "attention_fusion.gate_network.0.bias",
    "attention_fusion.gate_network.0.weight",
    "attention_fusion.gate_network.3.bias",
    "attention_fusion.gate_network.3.weight",
    "attention_fusion.projection.0.bias",
    "attention_fusion.projection.0.weight",
    "attention_fusion.projection.3.bias",
    "attention_fusion.projection.3.weight"
  ],
  "attention_fusion_present": true
}



745

## **9 — Nạp UniFashion + Attention-Gated Fusion**

Khi đánh giá, mô hình UniFashion được khởi tạo từ backbone và pretrained weights chính thức. Sau đó, các tham số đã được fine-tune ở Tuần 41, bao gồm Attention-Gated Fusion, được nạp từ best_monitor.pt.

Vì vậy, mô hình dùng trong Tuần 42 chính là mô hình đã được huấn luyện trước đó, không phải một Attention-Gated Fusion được khởi tạo ngẫu nhiên lại.

In [ ]:
# Tạo script evaluation độc lập có checkpoint/resume để dùng đúng Python + dependencies đã ghim.
eval_script = r"""
import argparse, hashlib, json, math, os, sys, time, traceback, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader

ROOT = Path("/content/khoaluantotnghiep")
sys.path.insert(0, str(ROOT))

from src.unifashion_runtime import build_model
from src.unifashion_training import load_trainable_state
from src.unifashion_sanity_data import preprocess, clean_caption
from scripts.train_unifashion_sanity import autocast_context


class ImageDataset(Dataset):
    def __init__(self, names, images_dir, transform):
        self.names = list(names)
        self.images_dir = Path(images_dir)
        self.transform = transform

    def __len__(self):
        return len(self.names)

    def __getitem__(self, idx):
        p = self.images_dir / self.names[idx]
        if not p.is_file():
            raise FileNotFoundError(p)
        with Image.open(p) as im:
            return self.transform(im.convert("RGB"))


def find_image_dir(images_dir, names):
    images_dir = Path(images_dir)
    direct = sum((images_dir / x).is_file() for x in names[:min(100, len(names))])
    if direct:
        return images_dir
    raise RuntimeError(
        "Images are not directly inside IMAGES_DIR. "
        "Point IMAGES_DIR to the folder containing the image files."
    )


def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def read_csv_checked(path, label, required_columns):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"{label} not found: {path}")
    if path.stat().st_size == 0:
        raise RuntimeError(
            f"{label} is 0 bytes: {path}. Input preparation was interrupted; rerun notebook from Cell 7.3."
        )
    try:
        df = pd.read_csv(path)
    except pd.errors.EmptyDataError as error:
        raise RuntimeError(
            f"{label} has no CSV columns: {path}. Rerun notebook from Cell 7.3."
        ) from error
    if df.empty:
        raise RuntimeError(f"{label} has zero rows: {path}")
    missing = set(required_columns) - set(df.columns)
    if missing:
        raise RuntimeError(f"{label} missing columns: {sorted(missing)}")
    return df


def atomic_json(path, payload):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(tmp, path)


def atomic_torch_save(path, payload):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, tmp)
    os.replace(tmp, path)


def atomic_csv(path, df):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)


def write_status(output, **kwargs):
    status_path = Path(output) / "status.json"
    old = {}
    if status_path.is_file():
        try:
            old = json.loads(status_path.read_text(encoding="utf-8"))
        except Exception:
            old = {}
    old.update(kwargs)
    old["updated_at_unix"] = time.time()
    atomic_json(status_path, old)


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--project", type=Path, required=True)
    p.add_argument("--upstream", type=Path, required=True)
    p.add_argument("--assets", type=Path, required=True)
    p.add_argument("--checkpoint", type=Path, required=True)
    p.add_argument("--images-dir", type=Path, required=True)
    p.add_argument("--captions", type=Path, required=True)
    p.add_argument("--gallery-csv", type=Path, required=True)
    p.add_argument("--queries-csv", type=Path, required=True)
    p.add_argument("--output", type=Path, required=True)
    p.add_argument("--batch-size", type=int, default=2)
    p.add_argument("--gallery-chunk-size", type=int, default=256)
    p.add_argument("--query-chunk-size", type=int, default=20)
    args = p.parse_args()

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA required")
    if args.batch_size < 1 or args.gallery_chunk_size < 1 or args.query_chunk_size < 1:
        raise ValueError("batch/chunk sizes must be positive")

    args.output.mkdir(parents=True, exist_ok=True)
    gallery_cache = args.output / "gallery_cache"
    query_cache = args.output / "query_cache"
    gallery_cache.mkdir(exist_ok=True)
    query_cache.mkdir(exist_ok=True)

    # Đọc và kiểm tra input TRƯỚC fingerprint. Không bao giờ fingerprint một CSV rỗng/hỏng.
    gallery_required = {
        "image", "gender", "category", "sleeve", "neckline",
        "upper_fabric", "upper_pattern"
    }
    query_required = {
        "tier", "reference", "target_seed", "text", "gender", "category",
        "sleeve", "neckline", "upper_fabric", "upper_pattern", "positive_count"
    }
    gallery = read_csv_checked(args.gallery_csv, "gallery CSV", gallery_required)
    queries = read_csv_checked(args.queries_csv, "queries CSV", query_required)
    captions = json.loads(args.captions.read_text(encoding="utf-8"))

    names = gallery["image"].astype(str).tolist()
    if len(names) != len(set(names)):
        raise RuntimeError("gallery CSV contains duplicate image names")
    if queries["reference"].astype(str).duplicated().all() and len(queries) > 1:
        print("WARNING: many repeated references may be valid, continuing", flush=True)
    if queries["text"].astype(str).str.strip().eq("").any():
        raise RuntimeError("queries CSV contains empty text")

    images_dir = find_image_dir(args.images_dir, names)

    # Fingerprint v3 tách gallery cache và query cache.
    # Đổi query metadata không làm mất gallery embeddings đã trích hàng giờ.
    gallery_semantic = {
        "checkpoint_sha256": sha256_file(args.checkpoint),
        "gallery_csv_sha256": sha256_file(args.gallery_csv),
        "captions_sha256": sha256_file(args.captions),
    }
    query_semantic = {
        **gallery_semantic,
        "queries_csv_sha256": sha256_file(args.queries_csv),
    }
    fingerprint = {
        "schema": 3,
        "gallery_semantic": gallery_semantic,
        "query_semantic": query_semantic,
        "layout": {
            "batch_size": args.batch_size,
            "gallery_chunk_size": args.gallery_chunk_size,
            "query_chunk_size": args.query_chunk_size,
        },
    }
    fp_path = args.output / "resume_fingerprint.json"

    def previous_parts(old_fp):
        if not isinstance(old_fp, dict):
            return {}, {}
        if old_fp.get("schema") == 3:
            return old_fp.get("gallery_semantic", {}), old_fp.get("query_semantic", {})
        sem = old_fp.get("semantic", old_fp)
        if not isinstance(sem, dict):
            return {}, {}
        old_gallery = {
            "checkpoint_sha256": sem.get("checkpoint_sha256"),
            "gallery_csv_sha256": sem.get("gallery_csv_sha256"),
            "captions_sha256": sem.get("captions_sha256"),
        }
        old_query = {**old_gallery, "queries_csv_sha256": sem.get("queries_csv_sha256")}
        return old_gallery, old_query

    if fp_path.is_file():
        try:
            old_fp = json.loads(fp_path.read_text(encoding="utf-8"))
        except Exception:
            old_fp = {}
        old_gallery, old_query = previous_parts(old_fp)

        gallery_same = old_gallery == gallery_semantic
        query_same = old_query == query_semantic

        if gallery_same and query_same:
            print("RESUME FINGERPRINT: gallery/query compatible", flush=True)
        elif gallery_same and not query_same:
            # Gallery embeddings vẫn hợp lệ; chỉ cất query cache cũ nếu có.
            query_progress = list(query_cache.glob("query_*.done.json")) + list(query_cache.glob("query_*.csv"))
            if query_progress:
                stamp = time.strftime("%Y%m%d_%H%M%S")
                backup = args.output / f"query_cache_backup_{stamp}"
                shutil.move(str(query_cache), str(backup))
                query_cache.mkdir(exist_ok=True)
                print(f"RESUME: query input changed; kept gallery cache, old query cache -> {backup}", flush=True)
            else:
                print("RESUME: query input changed; gallery cache retained", flush=True)
        else:
            # Gallery semantic input thay đổi: cache embeddings cũ không còn an toàn.
            progress = list(gallery_cache.glob("gallery_*.pt")) + list(query_cache.glob("query_*"))
            if progress:
                stamp = time.strftime("%Y%m%d_%H%M%S")
                backup = args.output / f"incompatible_cache_backup_{stamp}"
                backup.mkdir(parents=True, exist_ok=False)
                for d in (gallery_cache, query_cache):
                    if d.exists():
                        shutil.move(str(d), str(backup / d.name))
                gallery_cache.mkdir(exist_ok=True)
                query_cache.mkdir(exist_ok=True)
                print(f"RESUME: gallery semantic changed; old caches preserved at {backup}", flush=True)
            else:
                print("RESUME: stale fingerprint without completed cache -> refreshed", flush=True)
        atomic_json(fp_path, fingerprint)
    else:
        atomic_json(fp_path, fingerprint)
        print("RESUME FINGERPRINT: created", flush=True)

    missing_caps = [x for x in names if x not in captions]
    if missing_caps:
        raise RuntimeError(f"Gallery has {len(missing_caps)} images without captions")

    total_gallery = len(names)
    total_queries = len(queries)
    print(f"TOTAL GALLERY = {total_gallery}", flush=True)
    print(f"TOTAL QUERIES = {total_queries}", flush=True)
    print(f"QUERY CSV BYTES = {args.queries_csv.stat().st_size}", flush=True)

    print("PHASE checkpoint_load", flush=True)
    try:
        saved = torch.load(args.checkpoint, map_location="cpu", weights_only=True, mmap=True)
    except Exception as error:
        print(f"Checkpoint mmap load failed ({type(error).__name__}: {error}); retry without mmap", flush=True)
        saved = torch.load(args.checkpoint, map_location="cpu", weights_only=True)
    config = dict(saved["config"])
    checkpoint_meta = {
        "epoch": saved.get("epoch"),
        "monitor_loss": saved.get("monitor_loss"),
        "selection": saved.get("selection"),
    }
    fusion_keys = [k for k in saved["model"] if k.startswith("attention_fusion.")]
    if not fusion_keys:
        raise RuntimeError("No attention_fusion weights in checkpoint")

    device = torch.device("cuda")
    print("PHASE model_build", flush=True)
    model, build_report = build_model(args.upstream, args.assets, config, device)
    load_trainable_state(model, saved["model"])
    del saved
    model.eval()
    print("MODEL READY", flush=True)

    transform = preprocess(config)
    t0 = time.monotonic()

    # ------------------------------------------------------------------
    # 1) GALLERY: cache theo chunk. Mỗi chunk hoàn tất được lưu ngay lên Drive.
    # ------------------------------------------------------------------
    gallery_image_cpu = None
    gallery_caption_cpu = None
    n_gallery_chunks = math.ceil(total_gallery / args.gallery_chunk_size)
    write_status(
        args.output,
        status="gallery_running",
        retrieval_evaluated=False,
        gallery_size=total_gallery,
        queries=total_queries,
        gallery_completed=0,
        query_completed=0,
    )

    with torch.inference_mode():
        for chunk_idx, start in enumerate(range(0, total_gallery, args.gallery_chunk_size), 1):
            end = min(start + args.gallery_chunk_size, total_gallery)
            chunk_names = names[start:end]
            chunk_path = gallery_cache / f"gallery_{start:06d}_{end-1:06d}.pt"

            if chunk_path.is_file():
                chunk = torch.load(chunk_path, map_location="cpu", weights_only=True)
                if chunk.get("names") != chunk_names:
                    raise RuntimeError(f"Gallery cache mismatch: {chunk_path}")
                print(
                    f"GALLERY {end}/{total_gallery} | chunk {chunk_idx}/{n_gallery_chunks} | RESUME CACHE",
                    flush=True,
                )
            else:
                loader = DataLoader(
                    ImageDataset(chunk_names, images_dir, transform),
                    batch_size=args.batch_size,
                    shuffle=False,
                    num_workers=0,
                )
                image_feats, caption_feats = [], []
                local_seen = 0

                for batch in loader:
                    ids = chunk_names[local_seen:local_seen + len(batch)]
                    with autocast_context(device, config["precision"]):
                        tf, _ = model.extract_target_features(batch.to(device))
                        cf = model.extract_caption_features(
                            [clean_caption(captions[x]) for x in ids]
                        )
                    image_feats.append(tf.float().cpu())
                    caption_feats.append(cf.float().cpu())
                    local_seen += len(batch)

                    absolute_seen = start + local_seen
                    if absolute_seen == 1 or absolute_seen % 50 == 0 or absolute_seen == total_gallery:
                        pct = 100.0 * absolute_seen / max(1, total_gallery)
                        print(
                            f"GALLERY {absolute_seen}/{total_gallery} | {pct:6.2f}% | "
                            f"chunk {chunk_idx}/{n_gallery_chunks}",
                            flush=True,
                        )

                chunk = {
                    "names": chunk_names,
                    "image_features": torch.cat(image_feats),
                    "caption_features": torch.cat(caption_feats),
                }
                atomic_torch_save(chunk_path, chunk)
                print(
                    f"GALLERY CHECKPOINT SAVED {end}/{total_gallery} -> {chunk_path.name}",
                    flush=True,
                )

            if len(chunk["names"]) != end - start:
                raise RuntimeError(f"Gallery cache row-count mismatch: {chunk_path}")
            if len(chunk["image_features"]) != end - start or len(chunk["caption_features"]) != end - start:
                raise RuntimeError(f"Gallery cache feature-count mismatch: {chunk_path}")

            if gallery_image_cpu is None:
                gallery_image_cpu = torch.empty(
                    (total_gallery,) + tuple(chunk["image_features"].shape[1:]),
                    dtype=chunk["image_features"].dtype,
                )
                gallery_caption_cpu = torch.empty(
                    (total_gallery,) + tuple(chunk["caption_features"].shape[1:]),
                    dtype=chunk["caption_features"].dtype,
                )
            gallery_image_cpu[start:end].copy_(chunk["image_features"])
            gallery_caption_cpu[start:end].copy_(chunk["caption_features"])
            del chunk

            write_status(
                args.output,
                status="gallery_running",
                gallery_completed=end,
                gallery_chunks_completed=chunk_idx,
                gallery_chunks_total=n_gallery_chunks,
                query_completed=0,
            )

    if gallery_image_cpu is None or gallery_caption_cpu is None:
        raise RuntimeError("Gallery cache is empty")
    if len(gallery_image_cpu) != total_gallery or len(gallery_caption_cpu) != total_gallery:
        raise RuntimeError("Gallery cache merge size mismatch")

    print("PHASE gallery_to_gpu", flush=True)
    target_features = gallery_image_cpu.to(device)
    target_captions = gallery_caption_cpu.to(device)
    del gallery_image_cpu, gallery_caption_cpu
    torch.cuda.empty_cache()

    write_status(
        args.output,
        status="query_running",
        gallery_completed=total_gallery,
        query_completed=0,
    )
    print("GALLERY COMPLETE. Bắt đầu query scoring.", flush=True)

    # ------------------------------------------------------------------
    # 2) QUERY: lưu kết quả theo chunk. Chunk đã xong sẽ được bỏ qua khi resume.
    # ------------------------------------------------------------------
    name_to_idx = {n: i for i, n in enumerate(names)}

    # Dựng ground-truth lookup một lần, tránh quét 40k+ gallery cho từng query.
    tier_cols = {
        "fabric_pattern": ["upper_fabric", "upper_pattern"],
        "sleeve_fabric_pattern": ["sleeve", "upper_fabric", "upper_pattern"],
        "detailed": ["sleeve", "neckline", "upper_fabric", "upper_pattern"],
    }
    positive_lookup = {}
    for tier_name, attrs in tier_cols.items():
        grouped = gallery.groupby(["gender", "category"] + attrs, sort=False).groups
        tier_map = {}
        for raw_key, indices in grouped.items():
            if not isinstance(raw_key, tuple):
                raw_key = (raw_key,)
            key = (str(raw_key[0]), str(raw_key[1])) + tuple(int(x) for x in raw_key[2:])
            tier_map[key] = tuple(int(i) for i in indices)
        positive_lookup[tier_name] = tier_map
    print("GROUND TRUTH LOOKUP READY", flush=True)

    n_query_chunks = math.ceil(total_queries / args.query_chunk_size)

    with torch.inference_mode():
        for chunk_idx, start in enumerate(range(0, total_queries, args.query_chunk_size), 1):
            end = min(start + args.query_chunk_size, total_queries)
            chunk_rows = queries.iloc[start:end].reset_index(drop=True)
            result_path = query_cache / f"query_{start:06d}_{end-1:06d}.csv"
            done_path = query_cache / f"query_{start:06d}_{end-1:06d}.done.json"

            if done_path.is_file() and result_path.is_file():
                try:
                    done = json.loads(done_path.read_text(encoding="utf-8"))
                    cached_rows = len(pd.read_csv(result_path))
                    valid_done = (
                        done.get("start") == start
                        and done.get("end") == end
                        and cached_rows == 2 * (end - start)
                    )
                except Exception:
                    valid_done = False
                if valid_done:
                    print(
                        f"QUERY {end}/{total_queries} | chunk {chunk_idx}/{n_query_chunks} | RESUME CACHE",
                        flush=True,
                    )
                    write_status(
                        args.output,
                        status="query_running",
                        gallery_completed=total_gallery,
                        query_completed=end,
                        query_chunks_completed=chunk_idx,
                        query_chunks_total=n_query_chunks,
                    )
                    continue
                print(f"QUERY CACHE INVALID -> recompute {start}:{end}", flush=True)

            qloader = DataLoader(
                ImageDataset(chunk_rows["reference"].tolist(), images_dir, transform),
                batch_size=args.batch_size,
                shuffle=False,
                num_workers=0,
            )
            chunk_records = []
            local_seen = 0

            for batch in qloader:
                rows = chunk_rows.iloc[local_seen:local_seen + len(batch)]
                texts = [clean_caption(x) for x in rows["text"].tolist()]
                ref = model.ln_vision(model.visual_encoder(batch.to(device)))

                with autocast_context(device, config["precision"]):
                    combined, visual = model.inference(
                        ref, target_features, texts, target_captions
                    )

                visual = visual.reshape(len(rows), total_gallery).float().cpu().numpy()
                combined = combined.reshape(len(rows), total_gallery).float().cpu().numpy()

                for local_i, (_, row) in enumerate(rows.iterrows()):
                    global_query_index = start + local_seen + local_i
                    tier = str(row["tier"])
                    if tier not in tier_cols:
                        raise RuntimeError(f"Unknown query tier: {tier}")
                    key = (str(row["gender"]), str(row["category"])) + tuple(
                        int(row[attr]) for attr in tier_cols[tier]
                    )
                    pos_idx = set(positive_lookup[tier].get(key, ()))
                    ref_idx = name_to_idx.get(str(row["reference"]))
                    if ref_idx is not None:
                        pos_idx.discard(ref_idx)
                    if not pos_idx:
                        raise RuntimeError(
                            f"Query {global_query_index+1} has no positive inside gallery"
                        )

                    declared_count = int(row["positive_count"])
                    if declared_count != len(pos_idx):
                        # Reference được chọn khác thuộc tính query nên bình thường không nằm trong positives.
                        # Cho phép chênh 1 chỉ để phòng trường hợp protocol tương lai cho reference trùng signature.
                        if abs(declared_count - len(pos_idx)) > 1:
                            raise RuntimeError(
                                f"Query {global_query_index+1} positive_count mismatch: "
                                f"declared={declared_count}, reconstructed={len(pos_idx)}"
                            )

                    for score_name, matrix in [
                        ("image_score", visual),
                        ("image_caption_score", combined),
                    ]:
                        scores = matrix[local_i].copy()
                        if ref_idx is not None:
                            scores[ref_idx] = -np.inf
                        order = np.argsort(-scores, kind="stable")
                        ranks = [int(np.flatnonzero(order == j)[0]) + 1 for j in pos_idx]
                        best_rank = min(ranks)
                        rec = {
                            "query_index": global_query_index + 1,
                            "tier": row["tier"],
                            "reference": row["reference"],
                            "target_seed": row["target_seed"],
                            "text": row["text"],
                            "gender": row["gender"],
                            "category": row["category"],
                            "positive_count": len(pos_idx),
                            "score_type": score_name,
                            "best_positive_rank": best_rank,
                            "top50": json.dumps([names[j] for j in order[:50]]),
                        }
                        for k in (1, 5, 10, 50):
                            rec[f"hit@{k}"] = int(best_rank <= k)
                        chunk_records.append(rec)

                    # Tiến độ chính xác theo query, kiểu 1/3000, 2/3000, ...
                    done_queries = global_query_index + 1
                    pct = 100.0 * done_queries / max(1, total_queries)
                    print(
                        f"QUERY {done_queries}/{total_queries} | {pct:6.2f}% | "
                        f"tier={row['tier']} | ref={row['reference']}",
                        flush=True,
                    )

                local_seen += len(batch)

            chunk_df = pd.DataFrame(chunk_records)
            atomic_csv(result_path, chunk_df)
            atomic_json(done_path, {
                "start": start,
                "end": end,
                "query_count": end - start,
                "record_count": len(chunk_df),
            })
            write_status(
                args.output,
                status="query_running",
                gallery_completed=total_gallery,
                query_completed=end,
                query_chunks_completed=chunk_idx,
                query_chunks_total=n_query_chunks,
            )
            print(
                f"QUERY CHECKPOINT SAVED {end}/{total_queries} -> {result_path.name}",
                flush=True,
            )

    # ------------------------------------------------------------------
    # 3) MERGE + METRICS: chỉ chạy khi mọi query chunk đã hoàn tất.
    # ------------------------------------------------------------------
    result_files = []
    for start in range(0, total_queries, args.query_chunk_size):
        end = min(start + args.query_chunk_size, total_queries)
        result_path = query_cache / f"query_{start:06d}_{end-1:06d}.csv"
        done_path = query_cache / f"query_{start:06d}_{end-1:06d}.done.json"
        if not done_path.is_file() or not result_path.is_file():
            raise RuntimeError(f"Missing completed query chunk: {start}:{end}")
        result_files.append(result_path)

    pred = pd.concat([pd.read_csv(x) for x in result_files], ignore_index=True)
    expected_records = total_queries * 2
    if len(pred) != expected_records:
        raise RuntimeError(
            f"Expected {expected_records} prediction records, got {len(pred)}"
        )
    atomic_csv(args.output / "retrieval_per_query.csv", pred)

    summary = (
        pred.groupby(["score_type", "tier"])
        .agg(
            queries=("reference", "size"),
            mean_positive_count=("positive_count", "mean"),
            mean_best_rank=("best_positive_rank", "mean"),
            **{f"Recall@{k}": (f"hit@{k}", "mean") for k in (1, 5, 10, 50)}
        )
        .reset_index()
    )
    for k in (1, 5, 10, 50):
        summary[f"Recall@{k}"] *= 100.0
    atomic_csv(args.output / "metrics.csv", summary)

    provenance = {
        "checkpoint": str(args.checkpoint),
        "checkpoint_epoch": checkpoint_meta["epoch"],
        "checkpoint_monitor_loss": checkpoint_meta["monitor_loss"],
        "checkpoint_selection": checkpoint_meta["selection"],
        "attention_fusion_keys": fusion_keys,
        "gallery_size": total_gallery,
        "queries": total_queries,
        "config": config,
        "seconds_current_session": time.monotonic() - t0,
        "resume_enabled": True,
        "gallery_chunk_size": args.gallery_chunk_size,
        "query_chunk_size": args.query_chunk_size,
        "fingerprint": fingerprint,
        "note": "custom DeepFashion-MultiModal attribute-retrieval protocol; not an official benchmark",
    }
    atomic_json(args.output / "provenance.json", provenance)
    write_status(
        args.output,
        status="evaluation_completed",
        retrieval_evaluated=True,
        gallery_size=total_gallery,
        queries=total_queries,
        gallery_completed=total_gallery,
        query_completed=total_queries,
        gallery_chunks_completed=n_gallery_chunks,
        gallery_chunks_total=n_gallery_chunks,
        query_chunks_completed=n_query_chunks,
        query_chunks_total=n_query_chunks,
    )

    print("EVALUATION COMPLETED", flush=True)
    print(summary.to_string(index=False), flush=True)


if __name__ == "__main__":
    try:
        main()
    except Exception as error:
        tb = traceback.format_exc()
        try:
            if "--output" in sys.argv:
                output = Path(sys.argv[sys.argv.index("--output") + 1])
                output.mkdir(parents=True, exist_ok=True)
                (output / "last_error.txt").write_text(tb, encoding="utf-8")
                write_status(
                    output,
                    status="failed_resumable",
                    retrieval_evaluated=False,
                    error_type=type(error).__name__,
                    error=str(error),
                )
        except Exception:
            pass
        print(tb, file=sys.stderr, flush=True)
        raise
"""

EVAL_SCRIPT = Path("/content/evaluate_deepfashion_attribute_retrieval.py")
EVAL_SCRIPT.write_text(eval_script, encoding="utf-8")
print("Created:", EVAL_SCRIPT)
print("Resume support: gallery chunks + query chunks = ENABLED")

Created: /content/evaluate_deepfashion_attribute_retrieval.py
Resume support: gallery chunks + query chunks = ENABLED


### **9.1 — Lưu gallery CSV và kiểm tra vị trí ảnh**


In [ ]:
gallery_csv = OUT42 / f"gallery_{EVAL_MODE}.csv"
gallery_tmp = gallery_csv.with_suffix(gallery_csv.suffix + ".tmp")
gallery_df.to_csv(gallery_tmp, index=False)

check_gallery = pd.read_csv(gallery_tmp)
if check_gallery.empty:
    raise RuntimeError("gallery CSV rỗng. Dừng trước evaluation.")
if len(check_gallery) != len(gallery_df):
    raise RuntimeError(
        f"gallery CSV temp thiếu dòng: expected={len(gallery_df)}, read_back={len(check_gallery)}"
    )
if "image" not in check_gallery.columns:
    raise RuntimeError("gallery CSV thiếu cột image")
os.replace(gallery_tmp, gallery_csv)

query_csv = OUT42 / f"queries_{EVAL_MODE}.csv"
if not query_csv.is_file() or query_csv.stat().st_size == 0:
    raise RuntimeError(
        f"Query CSV không tồn tại hoặc rỗng: {query_csv}. Chạy lại Cell 7.3 trước."
    )
check_queries = pd.read_csv(query_csv)
if check_queries.empty:
    raise RuntimeError("Query CSV có header nhưng không có query. Dừng trước evaluation.")

# Kiểm tra 20 ảnh đầu có nằm trực tiếp trong images/ không.
sample_names = gallery_df["image"].head(20).tolist()
direct_hits = sum((IMAGES_DIR / n).is_file() for n in sample_names)

print("Gallery rows:", len(check_gallery), "| bytes:", gallery_csv.stat().st_size)
print("Query rows:", len(check_queries), "| bytes:", query_csv.stat().st_size)
print("Direct image hits:", direct_hits, "/", len(sample_names))
if direct_hits != len(sample_names):
    raise RuntimeError(
        "Ảnh không nằm trực tiếp trong IMAGES_DIR. Kiểm tra lại đường dẫn trước khi chạy model."
    )
print("PASS: gallery/query CSV hợp lệ và đường dẫn ảnh đúng.")

Gallery rows: 41571 | bytes: 13521608
Query rows: 3000 | bytes: 673262
Direct image hits: 20 / 20
PASS: gallery/query CSV hợp lệ và đường dẫn ảnh đúng.


### **9.2 — Nếu ảnh không nằm trực tiếp trong `images/`, dò thử cấu trúc**


In [ ]:
if direct_hits != len(sample_names):
    candidates = []
    for p in IMAGES_DIR.rglob(sample_names[0]):
        candidates.append(p)
        if len(candidates) >= 10:
            break
    print("Ví dụ vị trí tìm thấy:")
    for p in candidates:
        print(p)
    raise RuntimeError(
        "Dừng ở đây để tránh chạy GPU sai đường dẫn. "
        "Sửa IMAGES_DIR ở Section 3 thành folder chứa trực tiếp file JPG."
    )
print("Không cần chỉnh IMAGES_DIR.")

Không cần chỉnh IMAGES_DIR.


## **10 — Chạy retrieval evaluation**

Cell này chạy **full evaluation có checkpoint/resume**.

- Gallery được lưu theo chunk vào `evaluation_full/gallery_cache/`.
- Query được lưu theo chunk vào `evaluation_full/query_cache/`.
- Nếu Colab ngắt, **không xóa thư mục `evaluation_full`**. Mở runtime mới và `Run all`; script tự bỏ qua các chunk đã hoàn tất.
- Tiến độ query được in rõ theo dạng `QUERY 1/3000`, `QUERY 2/3000`, ...
- `status.json` được cập nhật trong lúc chạy, không phải chờ tới cuối mới có trạng thái.


In [ ]:
import json
import subprocess
from pathlib import Path

EVAL_OUT = OUT42 / f"evaluation_{EVAL_MODE}"
EVAL_OUT.mkdir(parents=True, exist_ok=True)

STATUS_PATH = EVAL_OUT / "status.json"
LOG_PATH = EVAL_OUT / "evaluation_live.log"

# Input gate ngay trước subprocess. Nếu query preparation bị ngắt thì dừng ở đây,
# không để subprocess tạo fingerprint từ file rỗng.
for label, path in [("gallery", gallery_csv), ("queries", query_csv)]:
    if not path.is_file():
        raise FileNotFoundError(f"{label} CSV không tồn tại: {path}")
    if path.stat().st_size == 0:
        raise RuntimeError(
            f"{label} CSV đang 0 byte: {path}. Chạy lại từ Cell 7.3 để ghi atomic file hợp lệ."
        )

pre_gallery = pd.read_csv(gallery_csv)
pre_queries = pd.read_csv(query_csv)
if pre_gallery.empty or pre_queries.empty:
    raise RuntimeError(
        f"Input rỗng trước evaluation: gallery={len(pre_gallery)}, queries={len(pre_queries)}"
    )
print(f"INPUT READY | gallery={len(pre_gallery)} | queries={len(pre_queries)}")
print(f"query CSV bytes={query_csv.stat().st_size}")

completed = False
if STATUS_PATH.is_file():
    try:
        status = json.loads(STATUS_PATH.read_text(encoding="utf-8"))
    except Exception:
        status = {}
    completed = status.get("status") == "evaluation_completed"
    if completed and (EVAL_OUT / "metrics.csv").is_file() and (EVAL_OUT / "retrieval_per_query.csv").is_file():
        print("Evaluation đã hoàn tất, bỏ qua chạy lại:", EVAL_OUT)
    else:
        completed = False
        print("Phát hiện evaluation chưa hoàn tất. Script sẽ RESUME từ cache hợp lệ.")
        print(json.dumps(status, indent=2, ensure_ascii=False))

if not completed:
    cmd = [
        str(PYTHON), "-B", "-u", str(EVAL_SCRIPT),
        "--project", str(PROJECT),
        "--upstream", str(UPSTREAM),
        "--assets", str(ASSETS),
        "--checkpoint", str(BEST_CKPT),
        "--images-dir", str(IMAGES_DIR),
        "--captions", str(CAPTIONS_JSON),
        "--gallery-csv", str(gallery_csv),
        "--queries-csv", str(query_csv),
        "--output", str(EVAL_OUT),
        "--batch-size", str(BATCH_SIZE),
        "--gallery-chunk-size", str(GALLERY_CHUNK_SIZE),
        "--query-chunk-size", str(QUERY_CHUNK_SIZE),
    ]

    print("\nCOMMAND:")
    print(" ".join(cmd))
    print("\nNếu Colab ngắt: runtime mới -> Run all. Cache hợp lệ sẽ được resume.")
    print("Live log:", LOG_PATH)
    print("=" * 80)

    with LOG_PATH.open("a", encoding="utf-8") as log:
        log.write("\n\n" + "=" * 80 + "\nNEW EVALUATION SESSION\n" + "=" * 80 + "\n")
        log.flush()

        proc = subprocess.Popen(
            cmd,
            cwd=str(PROJECT),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        returncode = proc.wait()

    print("\n" + "=" * 80)
    print("PROCESS RETURN CODE:", returncode)

    if returncode != 0:
        print("\nEvaluation bị dừng nhưng cache hoàn tất vẫn được giữ.")
        print("KHÔNG xóa evaluation_full.")
        print("\n100 DÒNG LOG CUỐI")
        try:
            log_lines = LOG_PATH.read_text(encoding="utf-8", errors="replace").splitlines()
            print("\n".join(log_lines[-100:]))
        except Exception as log_error:
            print("Không đọc được live log:", repr(log_error))

        error_path = EVAL_OUT / "last_error.txt"
        if error_path.is_file():
            print("\n===== TRACEBACK CHI TIẾT =====")
            print(error_path.read_text(encoding="utf-8", errors="replace"))

        raise RuntimeError(
            f"Evaluation dừng với return code {returncode}. "
            "Giữ nguyên evaluation_full và Run all lại để resume sau khi xử lý lỗi hiển thị phía trên."
        )

    print("\nEvaluation process kết thúc thành công")

INPUT READY | gallery=41571 | queries=3000
query CSV bytes=673262
Phát hiện evaluation chưa hoàn tất. Script sẽ RESUME từ cache hợp lệ.
{
  "status": "failed_resumable",
  "retrieval_evaluated": false,
  "error_type": "EmptyDataError",
  "error": "No columns to parse from file",
  "updated_at_unix": 1793622836.5288637
}

COMMAND:
/content/unifashion-py312/bin/python -B -u /content/evaluate_deepfashion_attribute_retrieval.py --project /content/khoaluantotnghiep --upstream /content/UniFashion-upstream --assets /content/drive/MyDrive/UniFashionAssets --checkpoint /content/drive/MyDrive/KHOALUAN/Tuần 41/UniFashionResults_run2/best_monitor.pt --images-dir /content/drive/MyDrive/KHOALUAN/deepfashion_mm/images --captions /content/drive/MyDrive/KHOALUAN/deepfashion_mm/captions.json --gallery-csv /content/drive/MyDrive/KHOALUAN/Tuần 42/DeepFashionAttributeRetrieval/gallery_full.csv --queries-csv /content/drive/MyDrive/KHOALUAN/Tuần 42/DeepFashionAttributeRetrieval/queries_full.csv --output /con

## **11 — Xem bảng metric**


In [ ]:
metrics_path = EVAL_OUT / "metrics.csv"
if not metrics_path.is_file():
    raise FileNotFoundError(metrics_path)

metrics = pd.read_csv(metrics_path)
display(metrics)

print("\nLƯU Ý:")
print("- Đây là custom attribute retrieval protocol.")
print("- Recall@K = tỷ lệ query có ít nhất một ảnh ground-truth trong Top-K.")
print("- Không so trực tiếp số này với FashionIQ paper nếu dataset/protocol khác.")

,score_type,tier,queries,mean_positive_count,mean_best_rank,Recall@1,Recall@5,Recall@10,Recall@50
0,image_caption_score,detailed,1000,221.066,109.848,29.8,55.2,65.5,84.3
1,image_caption_score,fabric_pattern,1000,1797.975,22.147,36.7,71.3,82.8,94.8
2,image_caption_score,sleeve_fabric_pattern,1000,685.468,50.950,37.8,62.5,73.8,90.3
3,image_score,detailed,1000,221.066,318.012,14.4,32.1,41.8,65.6
4,image_score,fabric_pattern,1000,1797.975,64.133,22.2,49.2,63.5,85.4
5,image_score,sleeve_fabric_pattern,1000,685.468,119.144,22.8,48.2,59.8,81.3



LƯU Ý:
- Đây là custom attribute retrieval protocol.
- Recall@K = tỷ lệ query có ít nhất một ảnh ground-truth trong Top-K.
- Không so trực tiếp số này với FashionIQ paper nếu dataset/protocol khác.


## **12 — Phân tích lỗi cơ bản**


In [ ]:
detail = pred[
    (pred["tier"] == "detailed") &
    (pred["score_type"] == "image_score")
].copy()

if len(detail):
    print("Detailed queries:", len(detail))
    for k in (1,5,10,50):
        print(f"Recall@{k}: {100*detail[f'hit@{k}'].mean():.2f}%")

    misses = detail[detail["hit@10"] == 0].sort_values("best_positive_rank")
    print("\nVí dụ query chưa hit@10:")
    display(misses[["reference","text","positive_count","best_positive_rank"]].head(10))
else:
    print("Không có detailed query trong kết quả.")

Detailed queries: 1000
Recall@1: 14.40%
Recall@5: 32.10%
Recall@10: 41.80%
Recall@50: 65.60%

Ví dụ query chưa hit@10:


,reference,text,positive_count,best_positive_rank
4174,WOMEN-Skirts-id_00007244-08_1_front.jpg,Find clothing with short-sleeve upper clothing...,54,11
5786,WOMEN-Tees_Tanks-id_00003055-02_2_side.jpg,"Find clothing with sleeveless upper clothing, ...",789,11
5606,WOMEN-Blouses_Shirts-id_00007432-03_4_full.jpg,"Find clothing with sleeveless upper clothing, ...",36,11
5742,WOMEN-Jackets_Coats-id_00003781-02_1_front.jpg,"Find clothing with long-sleeve upper clothing,...",96,11
5348,WOMEN-Rompers_Jumpsuits-id_00002100-02_7_addit...,"Find clothing with sleeveless upper clothing, ...",133,11
4360,WOMEN-Tees_Tanks-id_00006516-02_4_full.jpg,Find clothing with short-sleeve upper clothing...,14,11
5104,WOMEN-Dresses-id_00003138-02_7_additional.jpg,"Find clothing with long-sleeve upper clothing,...",16,11
5096,WOMEN-Sweaters-id_00000888-05_2_side.jpg,"Find clothing with long-sleeve upper clothing,...",252,11
5942,MEN-Tees_Tanks-id_00003425-04_3_back.jpg,"Find clothing with sleeveless upper clothing, ...",211,11
5164,WOMEN-Tees_Tanks-id_00007903-07_2_side.jpg,"Find clothing with sleeveless upper clothing, ...",789,11
